In [ ]:
# =============================================================================
# NOTEBOOK: Predictive Coordinated Volt-VAR and BESS Dispatch
#           for LV Voltage Regulation under High PV Penetration
# Implements the MPC formulation derived in the companion IEEE paper section.
# =============================================================================

# %%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 │ Install / Import packages + reproducibility seed
# ─────────────────────────────────────────────────────────────────────────────
# Installs required packages (run once), then imports everything needed
# by all subsequent cells. A fixed random seed is set for reproducibility.

# !pip install opendssdirect.py cvxpy osqp pandas numpy matplotlib openpyxl xlrd

import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless; swap to "TkAgg" for interactive
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cvxpy as cp
import opendssdirect as dss

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"CVXPY        : {cp.__version__}")
print(f"OpenDSSDirect: {dss.__version__}")


In [ ]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 │ CONFIG paths + file validation + output folder creation
# ─────────────────────────────────────────────────────────────────────────────
# All file paths live in one CONFIG dict. Edit ONLY this cell for your system.
# Every required file is validated; a clear error is raised if anything is missing.

CONFIG = {
    # ── Feeder ────────────────────────────────────────────────────────────────
    "feeder_dir"      : r"C:/LVTestCase",          # folder with Master*.dss
    "master_build"    : r"C:/LVTestCase/Master_build_only.dss",

    # ── Outputs from previous stages ──────────────────────────────────────────
    "out_dir"         : r"C:/LVTestCase/outputs",
    "placement_json"  : r"C:/LVTestCase/outputs/placement_rationale.json",
    "bus_catalog_csv" : r"C:/LVTestCase/outputs/bus_catalog.csv",
    "pv_mult_csv"     : r"C:/LVTestCase/outputs/pv_actual_mult.csv",   # 8760 rows
    "baseline_csv"    : r"C:/LVTestCase/outputs/baseline_annual.csv",
    "noder_csv"       : r"C:/LVTestCase/outputs/no_der_annual.csv",    # optional

    # ── Raw PV file (optional; set to None if unavailable) ────────────────────
    # Supported: .csv  |  .xlsx  |  "CSV disguised as .xls"
    # Required columns: timestamp, pv_actual_kw, pv_forecast_1h_kw
    "pv_raw_file"     : None,   # e.g. r"C:/data/pv_data.xlsx"

    # ── MPC hyperparameters ───────────────────────────────────────────────────
    "H"               : 12,      # horizon (hours)
    "dt"              : 1.0,     # time step (h)
    "T"               : 8760,    # annual simulation length

    # ── Voltage limits ────────────────────────────────────────────────────────
    "V_min"           : 0.95,
    "V_max"           : 1.05,

    # ── PV inverter ───────────────────────────────────────────────────────────
    "pv_rated_kw"     : 50.0,    # rated active power (kW)
    "S_inv_kva"       : 60.0,    # inverter apparent power rating (kVA)

    # ── BESS parameters ───────────────────────────────────────────────────────
    "bess_kw_rated"   : 30.0,    # max charge / discharge power (kW)
    "bess_kwh_rated"  : 60.0,    # energy capacity (kWh)
    "eta_ch"          : 0.92,    # charging efficiency
    "eta_dis"         : 0.92,    # discharging efficiency
    "soc_min"         : 0.10,
    "soc_max"         : 0.90,
    "soc_init"        : 0.50,    # initial SOC
    "pct_eff_charge"  : 92,      # OpenDSS %EffCharge (integer percent)
    "pct_eff_discharge": 92,     # OpenDSS %EffDischarge

    # ── MPC objective weights (w_V >> w_du >= w_B >> w_C) ────────────────────
    "w_V"             : 1e4,     # voltage violation slack penalty (dominant)
    "w_du"            : 1.0,     # control smoothness penalty
    "w_B"             : 0.5,     # BESS throughput / cycling penalty
    "w_C"             : 0.0,     # PV curtailment penalty (0 = disabled)

    # ── Sensitivity update frequency ──────────────────────────────────────────
    "sens_update_every": 1,      # re-linearise every N hours (1 = every step)
    "eps_perturb"     : 0.5,     # perturbation size for finite-difference (kW/kvar)

    # ── Ramp-rate limits (set to None to disable) ─────────────────────────────
    "dP_bess_max"     : None,    # kW/step
    "dQ_pv_max"       : None,    # kvar/step

    # ── Output paths ──────────────────────────────────────────────────────────
    "fig_dir"         : r"C:/LVTestCase/outputs/figures",
    "mpc_annual_csv"  : r"C:/LVTestCase/outputs/mpc_controlled_annual.csv",
    "mpc_summary_json": r"C:/LVTestCase/outputs/mpc_summary.json",
}

# ── Validate required files ────────────────────────────────────────────────
REQUIRED = [
    ("master_build",   CONFIG["master_build"]),
    ("placement_json", CONFIG["placement_json"]),
    ("bus_catalog_csv",CONFIG["bus_catalog_csv"]),
    ("pv_mult_csv",    CONFIG["pv_mult_csv"]),
    ("baseline_csv",   CONFIG["baseline_csv"]),
]
for label, path in REQUIRED:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"[CELL 1] Required file '{label}' not found:\n  {path}")
    print(f"  ✓  {label:20s}  {path}")

# Optional files
for label, key in [("no_der_csv","noder_csv"), ("pv_raw_file","pv_raw_file")]:
    p = CONFIG.get(key)
    print(f"  {'✓' if p and os.path.isfile(p) else '–'}  {label:20s}  {p or 'not provided'}")

# ── Create output directories ─────────────────────────────────────────────
os.makedirs(CONFIG["out_dir"],  exist_ok=True)
os.makedirs(CONFIG["fig_dir"],  exist_ok=True)
print(f"\nOutput dir  : {CONFIG['out_dir']}")
print(f"Figures dir : {CONFIG['fig_dir']}")


In [ ]:
 %%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 │ Load placement_rationale.json and bus_catalog.csv
# ─────────────────────────────────────────────────────────────────────────────
# Reads the bus placement decisions from the earlier screening stage.
# Extracts pv_bus and bess_bus that are used to place DER elements in Cell 5.

with open(CONFIG["placement_json"]) as f:
    placement = json.load(f)

pv_bus   = placement["pv_bus"].lower()
bess_bus = placement["bess_bus"].lower()
print(f"PV   bus : {pv_bus}")
print(f"BESS bus : {bess_bus}")
print(f"Rationale: {placement.get('rationale','(none)')}")

bus_cat = pd.read_csv(CONFIG["bus_catalog_csv"])
bus_cat.columns = bus_cat.columns.str.strip().str.lower()
print(f"\nBus catalog: {len(bus_cat)} buses")
print(bus_cat.head(3))

# Look up kV base for PV and BESS buses
def bus_kvbase(catalog_df, bus_name):
    row = catalog_df[catalog_df["bus"].str.lower() == bus_name.lower()]
    if len(row) == 0:
        return 0.416   # LV default (kV line-to-line)
    return float(row.iloc[0].get("kvbase", 0.416))

pv_kv   = bus_kvbase(bus_cat, pv_bus)
bess_kv = bus_kvbase(bus_cat, bess_bus)
print(f"\nPV   bus kV base : {pv_kv} kV")
print(f"BESS bus kV base : {bess_kv} kV")

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 │ Compile OpenDSS feeder (build-only master)
# ─────────────────────────────────────────────────────────────────────────────
# Compiles the feeder using absolute path to prevent "Redirect not found"
# errors. Sets the OpenDSS Datapath to the feeder directory so all relative
# Redirect/Compile calls inside Master_build_only.dss resolve correctly.

def compile_feeder(master_path: str, feeder_dir: str) -> None:
    """Compile the OpenDSS feeder and verify convergence."""
    master_abs = os.path.abspath(master_path)
    feeder_abs = os.path.abspath(feeder_dir)

    dss.Basic.ClearAll()
    dss.run_command(f'Set Datapath="{feeder_abs}"')
    dss.run_command(f'Compile "{master_abs}"')
    dss.run_command("Set VoltageBases=[11, 0.416]")
    dss.run_command("CalcVoltageBases")
    dss.run_command("Set MaxIter=100")
    dss.run_command("Set Tolerance=1e-8")
    dss.Solution.Solve()

    if not dss.Solution.Converged():
        raise RuntimeError("[CELL 3] Initial power flow did not converge!")
    print(f"  ✓ Feeder compiled and converged.")
    print(f"    Buses  : {dss.Circuit.NumBuses()}")
    print(f"    Nodes  : {dss.Circuit.NumNodes()}")

compile_feeder(CONFIG["master_build"], CONFIG["feeder_dir"])


In [ ]:
 %%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 │ Load PV data + create LoadShape.PV_ACT in OpenDSS
# ─────────────────────────────────────────────────────────────────────────────
# Supports three data paths in priority order:
#   1. Raw PV file (CSV / XLSX / XLS) with pv_actual_kw + pv_forecast_1h_kw
#   2. pv_actual_mult.csv × pv_rated_kw  (perfect-forecast fallback)
# The 8760-row multiplier array is also embedded into OpenDSS as LoadShape.PV_ACT
# so the PVSystem element can follow it during the annual simulation.

# ── Robust file loader ─────────────────────────────────────────────────────
def load_tabular(path: str) -> pd.DataFrame:
    """Load CSV / XLSX / 'CSV disguised as .xls' robustly."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".xlsx":
        return pd.read_excel(path, engine="openpyxl")
    if ext == ".xls":
        try:
            return pd.read_excel(path, engine="xlrd")
        except Exception:
            # Disguised CSV
            return pd.read_csv(path, sep=None, engine="python")
    return pd.read_csv(path, sep=None, engine="python")

# ── Attempt to load raw PV file ───────────────────────────────────────────
pv_raw_path = CONFIG.get("pv_raw_file")
pv_actual_kw   = None
pv_forecast_kw = None

if pv_raw_path and os.path.isfile(pv_raw_path):
    raw = load_tabular(pv_raw_path)
    raw.columns = raw.columns.str.strip().str.lower()
    if "pv_actual_kw" in raw.columns and "pv_forecast_1h_kw" in raw.columns:
        pv_actual_kw   = raw["pv_actual_kw"].values[:CONFIG["T"]].astype(float)
        pv_forecast_kw = raw["pv_forecast_1h_kw"].values[:CONFIG["T"]].astype(float)
        print("  ✓ Loaded raw PV file: actual + forecast columns found.")
    else:
        print("  ⚠ Raw PV file missing required columns; falling back to pv_actual_mult.")

# ── Fallback: pv_actual_mult × rated kW (perfect forecast) ───────────────
if pv_actual_kw is None:
    mult_df = pd.read_csv(CONFIG["pv_mult_csv"], header=None)
    mult    = mult_df.iloc[:, 0].values[:CONFIG["T"]].astype(float)
    mult    = np.clip(mult, 0.0, 1.0)
    pv_actual_kw   = mult * CONFIG["pv_rated_kw"]
    pv_forecast_kw = pv_actual_kw.copy()   # perfect forecast assumption
    print("  ✓ Perfect-forecast fallback: pv_forecast = pv_actual (from multipliers).")

# Ensure exactly T entries
pv_actual_kw   = np.resize(pv_actual_kw,   CONFIG["T"])
pv_forecast_kw = np.resize(pv_forecast_kw, CONFIG["T"])

# ── Embed LoadShape into OpenDSS ──────────────────────────────────────────
mult_norm = (pv_actual_kw / CONFIG["pv_rated_kw"]).clip(0, 1)
mult_str  = ",".join(f"{v:.6f}" for v in mult_norm)

dss.run_command(
    f'New LoadShape.PV_ACT npts={CONFIG["T"]} interval=1 '
    f'mult=({mult_str}) useactual=no'
)
print(f"  ✓ LoadShape.PV_ACT embedded ({CONFIG['T']} hourly points).")

# Quick sanity check
idx_peak = int(np.argmax(pv_actual_kw))
print(f"  Peak PV  : {pv_actual_kw.max():.2f} kW at hour {idx_peak}")
print(f"  Peak fcst: {pv_forecast_kw.max():.2f} kW at hour {int(np.argmax(pv_forecast_kw))}")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 │ Create PVSystem.PV_MAIN and Storage.BESS_MAIN; sanity-check solve
# ─────────────────────────────────────────────────────────────────────────────
# Adds the DER elements to the compiled feeder at the buses identified in
# Cell 2. Uses EDIT if the element already exists to avoid duplicate NEW errors.
# A sanity-check power flow is run to verify convergence with DERs present.

C = CONFIG   # shorthand

def _element_exists(elem_type: str, name: str) -> bool:
    all_names = [n.lower() for n in dss.Circuit.AllElementNames()]
    return f"{elem_type.lower()}.{name.lower()}" in all_names

# ── PVSystem.PV_MAIN ──────────────────────────────────────────────────────
pv_cmd = (
    f'New PVSystem.PV_MAIN '
    f'bus1={pv_bus} '
    f'kV={pv_kv:.4f} '
    f'kVA={C["S_inv_kva"]:.1f} '
    f'Pmpp={C["pv_rated_kw"]:.1f} '
    f'pf=1.0 '
    f'daily=PV_ACT '
    f'enabled=yes'
)
if _element_exists("PVSystem", "PV_MAIN"):
    dss.run_command(f'Edit PVSystem.PV_MAIN kVA={C["S_inv_kva"]:.1f} Pmpp={C["pv_rated_kw"]:.1f}')
    print("  EDIT  PVSystem.PV_MAIN (already exists)")
else:
    dss.run_command(pv_cmd)
    print("  NEW   PVSystem.PV_MAIN")

# ── Storage.BESS_MAIN ─────────────────────────────────────────────────────
bess_cmd = (
    f'New Storage.BESS_MAIN '
    f'bus1={bess_bus} '
    f'kV={bess_kv:.4f} '
    f'kWrated={C["bess_kw_rated"]:.1f} '
    f'kWhrated={C["bess_kwh_rated"]:.1f} '
    f'%stored={C["soc_init"]*100:.1f} '
    f'%EffCharge={C["pct_eff_charge"]} '
    f'%EffDischarge={C["pct_eff_discharge"]} '
    f'%reserve={C["soc_min"]*100:.1f} '
    f'State=IDLING '
    f'enabled=yes'
)
if _element_exists("Storage", "BESS_MAIN"):
    dss.run_command(
        f'Edit Storage.BESS_MAIN '
        f'kWrated={C["bess_kw_rated"]:.1f} kWhrated={C["bess_kwh_rated"]:.1f} '
        f'%EffCharge={C["pct_eff_charge"]} %EffDischarge={C["pct_eff_discharge"]}'
    )
    print("  EDIT  Storage.BESS_MAIN (already exists)")
else:
    dss.run_command(bess_cmd)
    print("  NEW   Storage.BESS_MAIN")

# ── Sanity solve ──────────────────────────────────────────────────────────
dss.run_command("Set Mode=Snapshot")
dss.run_command("Set Hour=0")
dss.Solution.Solve()
assert dss.Solution.Converged(), "[CELL 5] Sanity solve did not converge!"
vm = dss.Circuit.AllBusMagPu()
print(f"\n  ✓ Sanity solve converged.  Vmax={max(vm):.4f}  Vmin={min(v for v in vm if v>0):.4f}")



In [ ]:
 #─────────────────────────────────────────────────────────────────────────────
# CELL 6 │ Measurement helper functions
# ─────────────────────────────────────────────────────────────────────────────
# Pure Python functions that query OpenDSS after each Solve call.
# All measurements are in physical units (kW, kvar, pu).

def measure_voltages() -> np.ndarray:
    """Return all bus voltage magnitudes in pu (excluding zero-voltage slack nodes)."""
    v = np.array(dss.Circuit.AllBusMagPu(), dtype=float)
    return v[v > 0.01]   # filter floating/unconnected nodes

def measure_vmax_vmin():
    v = measure_voltages()
    return float(v.max()), float(v.min())

def measure_all_bus_voltages() -> dict:
    """Return {bus_name: Vpu} dict for all active buses."""
    names = dss.Circuit.AllBusNames()
    mags  = dss.Circuit.AllBusMagPu()
    return {n.lower(): float(m) for n, m in zip(names, mags) if m > 0.01}

def measure_soc() -> float:
    """Return SOC as a fraction [0,1] read from Storage.BESS_MAIN.%stored."""
    dss.run_command("? Storage.BESS_MAIN.%stored")
    val = dss.Properties.Value("").strip()
    try:
        return float(val) / 100.0
    except Exception:
        # Fallback: iterate via circuit element
        dss.Circuit.SetActiveElement("Storage.BESS_MAIN")
        props = dss.CktElement.Properties("pctStored")  # varies by version
        try:
            return float(props) / 100.0
        except Exception:
            return C["soc_init"]

def measure_losses_kw() -> float:
    """Return total feeder active losses in kW."""
    losses = dss.Circuit.Losses()   # [W, var]
    return float(losses[0]) / 1000.0

def measure_pv_kw() -> float:
    """Return actual PV active output (kW) from OpenDSS post-solve."""
    dss.Circuit.SetActiveElement("PVSystem.PV_MAIN")
    pwr = dss.CktElement.Powers()   # [P1, Q1, P2, Q2, ...]  watts per terminal
    # Terminal 1 active power (generator convention: negative = generating)
    return -float(pwr[0]) / 1000.0 if len(pwr) > 0 else 0.0

def measure_head_power_kw() -> float:
    """Return active power at feeder head (kW); positive = import from grid."""
    dss.Circuit.SetActiveElement("Vsource.Source")
    pwr = dss.CktElement.Powers()
    return float(pwr[0]) / 1000.0 if len(pwr) > 0 else 0.0

def actuate_pv(q_kvar: float) -> None:
    """Set PVSystem.PV_MAIN reactive power setpoint."""
    dss.run_command(f"Edit PVSystem.PV_MAIN kvar={q_kvar:.4f}")

def actuate_bess(p_kw: float) -> None:
    """
    Set Storage.BESS_MAIN dispatch.
    Sign convention: p_kw > 0 → DISCHARGING; p_kw < 0 → CHARGING; 0 → IDLING.
    """
    if abs(p_kw) < 1e-3:
        dss.run_command("Edit Storage.BESS_MAIN State=IDLING kW=0")
    elif p_kw > 0:
        dss.run_command(f"Edit Storage.BESS_MAIN State=DISCHARGING kW={p_kw:.4f}")
    else:
        dss.run_command(f"Edit Storage.BESS_MAIN State=CHARGING kW={abs(p_kw):.4f}")

def solve_hour(hour: int) -> bool:
    """Set hour, solve, return convergence flag."""
    dss.run_command(f"Set Mode=Snapshot")
    dss.run_command(f"Set Hour={hour}")
    dss.Solution.Solve()
    return dss.Solution.Converged()

print("  ✓ Measurement and actuation helpers defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 │ Sensitivity estimation via OpenDSS finite-difference perturbation
# ─────────────────────────────────────────────────────────────────────────────
# Implements Eq.(2)–(3) from the MPC formulation:
#
#   [S_P]_{i,BESS} = ( V_i(P_BESS + ε) − V_i(P_BESS) ) / ε
#   [S_Q]_{i,PV}   = ( V_i(Q_PV  + ε) − V_i(Q_PV)   ) / ε
#
# Returns S_P and S_Q as 1-D arrays indexed by bus order.
# The linearisation point is the current operating point (last solved state).

def estimate_sensitivities(hour: int,
                           q_pv_0: float,
                           p_bess_0: float,
                           eps: float = None) -> tuple:
    """
    Compute voltage sensitivity vectors at the current operating point.

    Parameters
    ----------
    hour      : current simulation hour
    q_pv_0    : current Q_PV setpoint (kvar)
    p_bess_0  : current P_BESS setpoint (kW, signed)
    eps       : perturbation size (kW / kvar)

    Returns
    -------
    V0   : np.ndarray  — base voltage vector (pu) for all active buses
    S_P  : np.ndarray  — dV/dP_BESS  (pu / kW)
    S_Q  : np.ndarray  — dV/dQ_PV    (pu / kvar)
    n_bus: int         — number of active buses
    """
    if eps is None:
        eps = CONFIG["eps_perturb"]

    # ── Base-point solve ───────────────────────────────────────────────────
    actuate_pv(q_pv_0)
    actuate_bess(p_bess_0)
    conv = solve_hour(hour)
    if not conv:
        # Return trivial sensitivities; controller uses previous matrices
        n = len(measure_voltages())
        return measure_voltages(), np.zeros(n), np.zeros(n), n

    V0 = measure_voltages()
    n  = len(V0)

    # ── Perturb P_BESS by +ε → S_P ────────────────────────────────────────
    actuate_bess(p_bess_0 + eps)
    solve_hour(hour)
    Vp = measure_voltages()
    if len(Vp) != n:
        Vp = V0.copy()
    S_P = (Vp - V0) / eps

    # ── Restore P_BESS; perturb Q_PV by +ε → S_Q ─────────────────────────
    actuate_bess(p_bess_0)
    actuate_pv(q_pv_0 + eps)
    solve_hour(hour)
    Vq = measure_voltages()
    if len(Vq) != n:
        Vq = V0.copy()
    S_Q = (Vq - V0) / eps

    # ── Restore to base point ─────────────────────────────────────────────
    actuate_pv(q_pv_0)
    actuate_bess(p_bess_0)
    solve_hour(hour)

    return V0, S_P, S_Q, n

# ── Example output at a representative high-PV hour ──────────────────────
_demo_hour = int(np.argmax(pv_actual_kw))
V0_demo, S_P_demo, S_Q_demo, nb_demo = estimate_sensitivities(
    _demo_hour, 0.0, 0.0
)
print(f"Sensitivity demo at hour {_demo_hour} (peak PV = {pv_actual_kw[_demo_hour]:.1f} kW)")
print(f"  Active buses : {nb_demo}")
print(f"  V0  range    : [{V0_demo.min():.4f}, {V0_demo.max():.4f}] pu")
print(f"  S_P range    : [{S_P_demo.min():.6f}, {S_P_demo.max():.6f}] pu/kW")
print(f"  S_Q range    : [{S_Q_demo.min():.6f}, {S_Q_demo.max():.6f}] pu/kvar")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 │ MPC optimization (CVXPY / OSQP) — exact match to formulation
# ─────────────────────────────────────────────────────────────────────────────
# Implements the full QP (Problem 23) exactly as stated in the IEEE formulation:
#
#   Decision vars  : Q_PV[k], P_BESS[k], P_plus[k], P_minus[k],
#                    z[k] (|P_BESS| aux), xi_plus[k], xi_minus[k]  for k=0..H-1
#   Aux state      : SOC[k]  for k=1..H
#
#   Objective      : w_V * Σ(ξ⁺² + ξ⁻²)        [C2 – voltage slack]
#                  + w_du * Σ||Δu||²             [control smoothness]
#                  + w_B  * Σ z[k]               [throughput / degradation]
#                  + w_C  * Σ curtailment[k]      [optional]
#
#   Constraints    : C1 voltage prediction (linear sensitivity)
#                    C2 soft voltage bounds
#                    C3 PV reactive capability (box)
#                    C4 BESS power bounds
#                    C5 P_BESS decomposition
#                    C6 SOC dynamics with efficiencies
#                    C7 SOC bounds
#                    C8 SOC initial condition
#                    C9 LP throughput auxiliary
#                    C10 ramp-rate (optional)

def build_and_solve_mpc(
        V0:         np.ndarray,   # current voltage vector (pu)  [n_bus]
        S_P:        np.ndarray,   # sensitivity dV/dP_BESS        [n_bus]
        S_Q:        np.ndarray,   # sensitivity dV/dQ_PV          [n_bus]
        soc_meas:   float,        # measured SOC at time t        [0,1]
        pv_fcst:    np.ndarray,   # PV forecast P̂_PV[t..t+H-1]  [H] kW
        pv_avail_0: float,        # actual available P_PV at t=0  (kW)
        q_prev:     float,        # previous Q_PV applied (kvar)
        p_prev:     float,        # previous P_BESS applied (kW)
        cfg:        dict,
) -> dict:
    """
    Build and solve the MPC QP for one receding-horizon step.

    Returns a dict with:
        Q_PV_opt    : optimal Q_PV sequence [H]  (kvar)
        P_BESS_opt  : optimal P_BESS sequence [H] (kW, signed)
        SOC_pred    : predicted SOC trajectory [H+1]
        slack_plus  : ξ⁺ at k=0 [n_bus]
        slack_minus : ξ⁻ at k=0 [n_bus]
        status      : solver status string
        solve_ms    : wall-clock solve time (ms)
    """
    H        = cfg["H"]
    dt       = cfg["dt"]
    n_bus    = len(V0)
    V_min    = cfg["V_min"]
    V_max    = cfg["V_max"]
    S_inv    = cfg["S_inv_kva"]
    P_rated  = cfg["pv_rated_kw"]
    P_ch_max = cfg["bess_kw_rated"]
    P_dis_max= cfg["bess_kw_rated"]
    E_rated  = cfg["bess_kwh_rated"]
    eta_ch   = cfg["eta_ch"]
    eta_dis  = cfg["eta_dis"]
    soc_min  = cfg["soc_min"]
    soc_max  = cfg["soc_max"]
    w_V      = cfg["w_V"]
    w_du     = cfg["w_du"]
    w_B      = cfg["w_B"]
    w_C      = cfg["w_C"]

    # ── Decision variables ─────────────────────────────────────────────────
    Q_PV    = cp.Variable(H, name="Q_PV")          # kvar
    P_BESS  = cp.Variable(H, name="P_BESS")        # kW (signed)
    P_plus  = cp.Variable(H, nonneg=True)           # discharge component ≥ 0
    P_minus = cp.Variable(H, nonpos=True)           # charge component ≤ 0
    z       = cp.Variable(H, nonneg=True)           # |P_BESS| auxiliary
    xi_p    = cp.Variable((H, n_bus), nonneg=True)  # ξ⁺ voltage slack
    xi_m    = cp.Variable((H, n_bus), nonneg=True)  # ξ⁻ voltage slack
    SOC_var = cp.Variable(H + 1, name="SOC")        # predicted SOC

    # ── Capability limits (C3): Q̄_PV[k] = sqrt(S_inv² − P̂_PV[k]²) ───────
    Q_bar = np.sqrt(np.maximum(S_inv**2 - np.clip(pv_fcst[:H], 0, S_inv)**2, 0.0))

    # ── Predicted voltage (C1): V̂[k] = V0 + S_P*(ΔP_BESS[k]+ΔP_PV[k]) + S_Q*ΔQ_PV[k]
    # ΔP_BESS[k] = P_BESS[k] − 0  (linearisation at P0=0 for simplicity; full
    #              rolling re-lin sets P0=p_prev, but 0 is conservative and safe)
    # ΔP_PV[k]   = pv_fcst[k] − pv_avail_0  (known disturbance)
    # ΔQ_PV[k]   = Q_PV[k] − 0
    delta_P_PV = pv_fcst[:H] - pv_avail_0   # [H], kW

    constraints = []

    # ── C2: Soft voltage bounds ────────────────────────────────────────────
    # V̂[k] ≈ V0 + S_P*(P_BESS[k] + dP_PV[k]) + S_Q*Q_PV[k]
    # Written row-by-row for each bus; vectorised over buses with broadcasting.
    for k in range(H):
        V_pred = (V0
                  + S_P * (P_BESS[k] + delta_P_PV[k])
                  + S_Q * Q_PV[k])               # shape: (n_bus,)
        constraints += [
            V_pred <= V_max + xi_p[k],            # ξ⁺ absorbs overvoltage
            V_pred >= V_min - xi_m[k],            # ξ⁻ absorbs undervoltage
        ]

    # ── C3: PV reactive capability ────────────────────────────────────────
    constraints += [
        Q_PV >= -Q_bar,
        Q_PV <=  Q_bar,
    ]

    # ── C4: BESS power bounds ─────────────────────────────────────────────
    constraints += [
        P_BESS >= -P_ch_max,
        P_BESS <=  P_dis_max,
    ]

    # ── C5: P_BESS decomposition ──────────────────────────────────────────
    constraints += [
        P_BESS == P_plus + P_minus,
    ]

    # ── C6: SOC dynamics with efficiencies (Eq. 13) ───────────────────────
    constraints += [SOC_var[0] == soc_meas]         # C8: initial condition
    for k in range(H):
        # SOC[k+1] = SOC[k] - (dt/E_rated)*(P_plus[k]/eta_dis + eta_ch*P_minus[k])
        # Note: P_minus[k] ≤ 0, so eta_ch * P_minus[k] decreases charge (increase SOC)
        # sign is correct: subtracting a negative term → SOC increases when charging
        constraints += [
            SOC_var[k + 1] == SOC_var[k]
                             - (dt / E_rated) * (P_plus[k] / eta_dis
                                                 + eta_ch * P_minus[k])
        ]

    # ── C7: SOC bounds ────────────────────────────────────────────────────
    constraints += [
        SOC_var >= soc_min,
        SOC_var <= soc_max,
    ]

    # ── C9: LP auxiliary for |P_BESS| ─────────────────────────────────────
    constraints += [
        z >= P_plus,
        z >= -P_minus,
    ]

    # ── C10: Ramp-rate constraints (optional) ─────────────────────────────
    dP_max = cfg.get("dP_bess_max")
    dQ_max = cfg.get("dQ_pv_max")
    if dP_max is not None:
        constraints += [
            P_BESS[0] - p_prev <= dP_max,
            p_prev - P_BESS[0] <= dP_max,
        ]
        for k in range(1, H):
            constraints += [
                P_BESS[k] - P_BESS[k-1] <= dP_max,
                P_BESS[k-1] - P_BESS[k] <= dP_max,
            ]
    if dQ_max is not None:
        constraints += [
            Q_PV[0] - q_prev <= dQ_max,
            q_prev - Q_PV[0] <= dQ_max,
        ]
        for k in range(1, H):
            constraints += [
                Q_PV[k] - Q_PV[k-1] <= dQ_max,
                Q_PV[k-1] - Q_PV[k] <= dQ_max,
            ]

    # ── Objective (Eq. 17–20) ─────────────────────────────────────────────
    # Term 1: Voltage violation penalty (quadratic)
    J_V  = w_V  * (cp.sum_squares(xi_p) + cp.sum_squares(xi_m))

    # Term 2: Control smoothness (quadratic Δu)
    du_Q = cp.hstack([Q_PV[0] - q_prev,   cp.diff(Q_PV)])
    du_P = cp.hstack([P_BESS[0] - p_prev, cp.diff(P_BESS)])
    J_du = w_du * (cp.sum_squares(du_Q) + cp.sum_squares(du_P))

    # Term 3: Battery throughput / cycling (L1, linearised via z)
    J_B  = w_B  * cp.sum(z)

    # Term 4: Curtailment penalty (optional; pv_avail − pv_delivered)
    # When no active curtailment variable, this evaluates on forecast deviation.
    J_C  = w_C  * cp.sum(cp.maximum(pv_fcst[:H] - (pv_fcst[:H] + 0), 0))  # = 0 here

    objective = cp.Minimize(J_V + J_du + J_B + J_C)
    problem   = cp.Problem(objective, constraints)

    # ── Solve ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    problem.solve(solver=cp.OSQP,
                  warm_starting=True,
                  eps_abs=1e-5,
                  eps_rel=1e-5,
                  max_iter=10000,
                  verbose=False)
    solve_ms = (time.perf_counter() - t0) * 1e3

    status = problem.status

    if status in ("optimal", "optimal_inaccurate"):
        Q_opt  = Q_PV.value
        P_opt  = P_BESS.value
        SOC_pr = SOC_var.value
    else:
        # ── Fallback: full reactive absorb, BESS idle ──────────────────────
        Q_opt  = -Q_bar                          # maximum reactive absorption
        P_opt  = np.zeros(H)
        soc_seq = [soc_meas]
        for k in range(H):
            soc_seq.append(soc_seq[-1])
        SOC_pr = np.array(soc_seq)
        print(f"    [MPC FALLBACK] status={status}, applying Q_absorb, P_BESS=0")

    return {
        "Q_PV_opt"   : Q_opt,
        "P_BESS_opt" : P_opt,
        "SOC_pred"   : SOC_pr,
        "slack_plus" : xi_p[0].value if xi_p.value is not None else np.zeros(n_bus),
        "slack_minus": xi_m[0].value if xi_m.value is not None else np.zeros(n_bus),
        "status"     : status,
        "solve_ms"   : solve_ms,
    }

print("  ✓ MPC solver function 'build_and_solve_mpc' defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 │ Closed-loop validation run (48–168 hours)
# ─────────────────────────────────────────────────────────────────────────────
# Runs the full receding-horizon MPC loop on a short window to validate
# correctness before committing to the 8760-hour annual run.
# Saves a CSV and prints KPI summary at the end.

def run_mpc_loop(t_start: int, t_end: int, cfg: dict,
                 label: str = "mpc_run") -> pd.DataFrame:
    """
    Closed-loop MPC simulation from hour t_start to t_end (exclusive).

    Steps per hour (Section 6 of formulation):
      1. Measure from OpenDSS
      2. Update PV forecast
      3. Estimate sensitivities (every cfg['sens_update_every'] hours)
      4. Solve MPC
      5. Apply first control action (receding-horizon principle)
      6. Advance OpenDSS power flow
      7. Log
    """
    H   = cfg["H"]
    T   = t_end - t_start
    out = []

    # ── Compile fresh feeder ──────────────────────────────────────────────
    compile_feeder(cfg["master_build"], cfg["feeder_dir"])
    # Re-create DER elements (compile wiped them)
    dss.run_command(
        f'New PVSystem.PV_MAIN bus1={pv_bus} kV={pv_kv:.4f} '
        f'kVA={cfg["S_inv_kva"]:.1f} Pmpp={cfg["pv_rated_kw"]:.1f} '
        f'pf=1.0 daily=PV_ACT enabled=yes'
    )
    dss.run_command(
        f'New Storage.BESS_MAIN bus1={bess_bus} kV={bess_kv:.4f} '
        f'kWrated={cfg["bess_kw_rated"]:.1f} kWhrated={cfg["bess_kwh_rated"]:.1f} '
        f'%stored={cfg["soc_init"]*100:.1f} '
        f'%EffCharge={cfg["pct_eff_charge"]} '
        f'%EffDischarge={cfg["pct_eff_discharge"]} '
        f'%reserve={cfg["soc_min"]*100:.1f} '
        f'State=IDLING enabled=yes'
    )
    # Re-embed LoadShape
    mult_str_loc = ",".join(f"{v:.6f}" for v in (pv_actual_kw / cfg["pv_rated_kw"]).clip(0,1))
    dss.run_command(
        f'New LoadShape.PV_ACT npts={cfg["T"]} interval=1 '
        f'mult=({mult_str_loc}) useactual=no'
    )
    dss.run_command(f"Edit PVSystem.PV_MAIN daily=PV_ACT")

    # ── State memory ──────────────────────────────────────────────────────
    soc_cur   = cfg["soc_init"]
    q_prev    = 0.0
    p_prev    = 0.0
    V0_cur    = None
    S_P_cur   = None
    S_Q_cur   = None

    for step, t in enumerate(range(t_start, t_end)):
        # ── Step 1: Measure ───────────────────────────────────────────────
        conv = solve_hour(t)
        if not conv:
            print(f"  [t={t}] Non-convergence; skipping MPC, holding setpoints.")
            out.append({
                "hour": t, "converged": 0,
                "vmax": np.nan, "vmin": np.nan,
                "q_pv_kvar": q_prev, "p_bess_kw": p_prev,
                "soc": soc_cur, "losses_kw": np.nan,
                "pv_actual_kw": pv_actual_kw[t],
                "pv_forecast_kw": pv_forecast_kw[t],
                "mpc_status": "no_conv", "solve_ms": 0.0,
            })
            continue

        vmax, vmin = measure_vmax_vmin()
        soc_cur    = measure_soc()
        losses     = measure_losses_kw()
        pv_out     = measure_pv_kw()
        head_p     = measure_head_power_kw()

        # ── Step 2: Forecast window ───────────────────────────────────────
        t_end_fcst = min(t + H, cfg["T"])
        fcst_window = pv_forecast_kw[t:t_end_fcst]
        if len(fcst_window) < H:
            fcst_window = np.pad(fcst_window, (0, H - len(fcst_window)),
                                 mode="edge")

        # ── Step 3: Sensitivity update ────────────────────────────────────
        if step % cfg["sens_update_every"] == 0:
            V0_cur, S_P_cur, S_Q_cur, _ = estimate_sensitivities(
                t, q_prev, p_prev, cfg["eps_perturb"]
            )

        # ── Step 4: Solve MPC ─────────────────────────────────────────────
        result = build_and_solve_mpc(
            V0=V0_cur, S_P=S_P_cur, S_Q=S_Q_cur,
            soc_meas=soc_cur,
            pv_fcst=fcst_window,
            pv_avail_0=pv_actual_kw[t],
            q_prev=q_prev, p_prev=p_prev,
            cfg=cfg,
        )

        # ── Step 5: Apply first action ────────────────────────────────────
        q_apply = float(result["Q_PV_opt"][0])
        p_apply = float(result["P_BESS_opt"][0])
        actuate_pv(q_apply)
        actuate_bess(p_apply)
        q_prev = q_apply
        p_prev = p_apply

        # ── Step 6: Re-solve with applied setpoints ───────────────────────
        conv2 = solve_hour(t)
        vmax2, vmin2 = measure_vmax_vmin()
        losses2      = measure_losses_kw()
        soc_cur      = measure_soc()
        pv_out2      = measure_pv_kw()
        head_p2      = measure_head_power_kw()

        # ── Step 7: Log ───────────────────────────────────────────────────
        row = {
            "hour"           : t,
            "converged"      : int(conv2),
            "vmax"           : vmax2,
            "vmin"           : vmin2,
            "v_over"         : int(vmax2 > cfg["V_max"]),
            "v_under"        : int(vmin2 < cfg["V_min"]),
            "q_pv_kvar"      : q_apply,
            "p_bess_kw"      : p_apply,
            "soc"            : soc_cur,
            "losses_kw"      : losses2,
            "pv_actual_kw"   : pv_actual_kw[t],
            "pv_forecast_kw" : pv_forecast_kw[t],
            "pv_delivered_kw": pv_out2,
            "head_p_kw"      : head_p2,
            "mpc_status"     : result["status"],
            "solve_ms"       : result["solve_ms"],
        }
        out.append(row)

        if step % 24 == 0:
            print(f"  t={t:5d} | Vmax={vmax2:.4f} Vmin={vmin2:.4f} | "
                  f"Q={q_apply:+7.2f} kvar | P={p_apply:+7.2f} kW | "
                  f"SOC={soc_cur:.3f} | {result['status']} ({result['solve_ms']:.1f} ms)")

    df = pd.DataFrame(out)
    return df

# ── Run validation window ─────────────────────────────────────────────────
_T_VAL_START = max(0, int(np.argmax(pv_actual_kw)) - 72)  # 72 h before peak
_T_VAL_END   = min(CONFIG["T"], _T_VAL_START + 168)        # 1-week window

print(f"\nValidation run: hours {_T_VAL_START}–{_T_VAL_END} ({_T_VAL_END-_T_VAL_START} steps)")
df_val = run_mpc_loop(_T_VAL_START, _T_VAL_END, CONFIG, label="validation")

val_csv = os.path.join(CONFIG["out_dir"], "mpc_validation_window.csv")
df_val.to_csv(val_csv, index=False)
print(f"\nValidation CSV saved: {val_csv}")

# ── KPI mini-summary ──────────────────────────────────────────────────────
n_over   = df_val["v_over"].sum()
n_under  = df_val["v_under"].sum()
worst_hi = df_val["vmax"].max()
worst_lo = df_val["vmin"].min()
mean_ms  = df_val["solve_ms"].mean()
print(f"\nValidation KPIs ({_T_VAL_END - _T_VAL_START} h):")
print(f"  Over-voltage  hours : {n_over}")
print(f"  Under-voltage hours : {n_under}")
print(f"  Worst Vmax          : {worst_hi:.4f} pu")
print(f"  Worst Vmin          : {worst_lo:.4f} pu")
print(f"  Mean solve time     : {mean_ms:.2f} ms/step")

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 │ Full-year MPC run (8760 hours) → mpc_controlled_annual.csv
# ─────────────────────────────────────────────────────────────────────────────
# Executes the complete annual horizon.  Sensitivities are updated every
# cfg['sens_update_every'] hours.  Progress is printed every 24 steps.
# Saves the full log to mpc_controlled_annual.csv (matching columns of
# baseline_annual.csv plus MPC-specific setpoint columns).

print("Starting full-year MPC run (8760 hours) ...")
t_full_start = time.perf_counter()

df_annual = run_mpc_loop(0, CONFIG["T"], CONFIG, label="annual")

t_full_end = time.perf_counter()
total_min  = (t_full_end - t_full_start) / 60.0

df_annual.to_csv(CONFIG["mpc_annual_csv"], index=False)
print(f"\n  ✓ Annual run complete in {total_min:.1f} min")
print(f"  Saved: {CONFIG['mpc_annual_csv']}")
print(df_annual.tail(3))

In [ ]:
#%%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 │ Comparison plots: No DER vs Baseline vs MPC
# ─────────────────────────────────────────────────────────────────────────────
# Generates 6 publication-ready figures and saves them as PNGs:
#   Fig 1: Vmax / Vmin time-series (annual)
#   Fig 2: Hourly over-voltage indicator + cumulative violation hours
#   Fig 3: SOC trajectory + P_BESS dispatch
#   Fig 4: Q_PV reactive dispatch
#   Fig 5: Network losses comparison
#   Fig 6: PV actual vs delivered (curtailment view)

# ── Load comparison datasets ──────────────────────────────────────────────
df_mpc  = df_annual.copy()
df_base = pd.read_csv(CONFIG["baseline_csv"])
df_base.columns = df_base.columns.str.strip().str.lower()

df_noder = None
if CONFIG.get("noder_csv") and os.path.isfile(CONFIG["noder_csv"]):
    df_noder = pd.read_csv(CONFIG["noder_csv"])
    df_noder.columns = df_noder.columns.str.strip().str.lower()

hours = np.arange(CONFIG["T"])

# ── Helper: safe column retrieval ────────────────────────────────────────
def safe_col(df, col, fill=np.nan):
    return df[col].values if col in df.columns else np.full(len(df), fill)

vmax_mpc  = safe_col(df_mpc,  "vmax")
vmin_mpc  = safe_col(df_mpc,  "vmin")
vmax_base = safe_col(df_base, "vmax")
vmin_base = safe_col(df_base, "vmin")
vmax_nd   = safe_col(df_noder, "vmax") if df_noder is not None else None
vmin_nd   = safe_col(df_noder, "vmin") if df_noder is not None else None

SAVEFIG = lambda name: plt.savefig(
    os.path.join(CONFIG["fig_dir"], name), dpi=150, bbox_inches="tight"
)

In [ ]:
#───────────────────────────────────────────────────────────────────────
# Fig 1: Vmax / Vmin time-series
# ────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
h_lim_hi  = np.full(CONFIG["T"], CONFIG["V_max"])
h_lim_lo  = np.full(CONFIG["T"], CONFIG["V_min"])

for ax, (y_m, y_b, y_n, title, lim) in zip(
    axes,
    [(vmax_mpc, vmax_base, vmax_nd, "Vmax (pu)", CONFIG["V_max"]),
     (vmin_mpc, vmin_base, vmin_nd, "Vmin (pu)", CONFIG["V_min"])]):

    if y_n is not None:
        ax.plot(hours, y_n, color="silver", lw=0.6, label="No DER")
    ax.plot(hours, y_b, color="steelblue", lw=0.7, alpha=0.8, label="Baseline DER")
    ax.plot(hours, y_m, color="darkorange",lw=0.8, alpha=0.9, label="MPC")
    ax.axhline(lim, color="red", ls="--", lw=1.2, label=f"Limit {lim} pu")
    ax.set_ylabel(title); ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, ls=":", alpha=0.5)

axes[1].set_xlabel("Hour of year")
fig.suptitle("Annual Voltage Envelope: No DER vs Baseline vs MPC", fontsize=11)
fig.tight_layout()
SAVEFIG("fig1_voltage_envelope.png")
plt.close()
print("  ✓ Fig 1 saved.")


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Fig 2: Over-voltage indicator + cumulative violation hours
# ────────────────────────────────────────────────────────────────────────
over_mpc  = (vmax_mpc  > CONFIG["V_max"]).astype(float)
over_base = (vmax_base > CONFIG["V_max"]).astype(float)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].fill_between(hours, over_base, alpha=0.4, color="steelblue", label="Baseline")
axes[0].fill_between(hours, over_mpc,  alpha=0.6, color="darkorange",label="MPC")
axes[0].set_ylabel("Over-voltage\n(binary)"); axes[0].legend(fontsize=8)
axes[0].grid(True, ls=":", alpha=0.5)

axes[1].plot(hours, np.cumsum(over_base), color="steelblue", label="Baseline cumul.")
axes[1].plot(hours, np.cumsum(over_mpc),  color="darkorange",label="MPC cumul.")
axes[1].set_ylabel("Cumul. violation\nhours"); axes[1].legend(fontsize=8)
axes[1].set_xlabel("Hour of year"); axes[1].grid(True, ls=":", alpha=0.5)

fig.suptitle("Over-voltage Events: Baseline vs MPC", fontsize=11)
fig.tight_layout()
SAVEFIG("fig2_violation_hours.png")
plt.close()
print("  ✓ Fig 2 saved.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Fig 3: SOC + P_BESS
# ────────────────────────────────────────────────────────────────────────
soc_mpc  = safe_col(df_mpc, "soc")
p_bess   = safe_col(df_mpc, "p_bess_kw", fill=0.0)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(hours, soc_mpc, color="seagreen", lw=0.8)
axes[0].axhline(CONFIG["soc_min"], color="red",  ls="--", lw=1)
axes[0].axhline(CONFIG["soc_max"], color="red",  ls="--", lw=1)
axes[0].set_ylabel("SOC (p.u.)"); axes[0].set_ylim(0, 1.05)
axes[0].grid(True, ls=":", alpha=0.5)

axes[1].fill_between(hours, p_bess,
    where=(p_bess > 0), interpolate=True, color="tomato",  alpha=0.7, label="Discharge")
axes[1].fill_between(hours, p_bess,
    where=(p_bess < 0), interpolate=True, color="royalblue",alpha=0.7, label="Charge")
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_ylabel("P_BESS (kW)"); axes[1].legend(fontsize=8)
axes[1].set_xlabel("Hour of year"); axes[1].grid(True, ls=":", alpha=0.5)

fig.suptitle("BESS: SOC and Active Power Dispatch (MPC)", fontsize=11)
fig.tight_layout()
SAVEFIG("fig3_bess_soc_dispatch.png")
plt.close()
print("  ✓ Fig 3 saved.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Fig 4: Q_PV reactive dispatch
# ────────────────────────────────────────────────────────────────────────
q_pv = safe_col(df_mpc, "q_pv_kvar", fill=0.0)

fig, ax = plt.subplots(figsize=(14, 3))
ax.fill_between(hours, q_pv,
    where=(q_pv > 0), interpolate=True, color="gold",     alpha=0.8, label="Capacitive (+)")
ax.fill_between(hours, q_pv,
    where=(q_pv < 0), interpolate=True, color="mediumpurple",alpha=0.8, label="Inductive (−)")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Q_PV (kvar)"); ax.set_xlabel("Hour of year")
ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)
fig.suptitle("PV Inverter Reactive Power Dispatch (MPC)", fontsize=11)
fig.tight_layout()
SAVEFIG("fig4_qpv_dispatch.png")
plt.close()
print("  ✓ Fig 4 saved.")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Fig 5: Losses comparison
# ────────────────────────────────────────────────────────────────────────
loss_mpc  = safe_col(df_mpc,  "losses_kw", fill=0.0)
loss_base = safe_col(df_base, "losses_kw", fill=0.0)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(hours[:len(loss_base)], loss_base, color="steelblue", lw=0.6,
        alpha=0.8, label="Baseline")
ax.plot(hours,                  loss_mpc,  color="darkorange", lw=0.7,
        alpha=0.9, label="MPC")
ax.set_ylabel("Losses (kW)"); ax.set_xlabel("Hour of year")
ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)
fig.suptitle("Feeder Active Losses: Baseline vs MPC", fontsize=11)
fig.tight_layout()
SAVEFIG("fig5_losses.png")
plt.close()
print("  ✓ Fig 5 saved.")

In [ ]:
 #────────────────────────────────────────────────────────────────────────
# Fig 6: PV available vs delivered
# ────────────────────────────────────────────────────────────────────────
pv_avail = pv_actual_kw
pv_deliv = safe_col(df_mpc, "pv_delivered_kw", fill=pv_actual_kw)

fig, ax = plt.subplots(figsize=(14, 3))
ax.fill_between(hours, pv_avail, label="Available", alpha=0.4, color="gold")
ax.fill_between(hours, pv_deliv, label="Delivered", alpha=0.7, color="darkorange")
ax.set_ylabel("PV Power (kW)"); ax.set_xlabel("Hour of year")
ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)
fig.suptitle("PV Available vs Delivered (curtailment view)", fontsize=11)
fig.tight_layout()
SAVEFIG("fig6_pv_curtailment.png")
plt.close()
print("  ✓ Fig 6 saved.")

In [ ]:
#  %%
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 │ IEEE-ready summary tables + mpc_summary.json
# ─────────────────────────────────────────────────────────────────────────────
# Computes all KPIs defined in Section 8.2 of the formulation and exports:
#   • a printed table for visual inspection
#   • mpc_summary.json (machine-readable, includes weights, horizon, solver info)
#   • mpc_summary_table.md  (paste directly into LaTeX / Markdown paper)

def compute_kpis(df: pd.DataFrame, tag: str, cfg: dict) -> dict:
    """Compute annual KPI set for one simulation run."""
    T  = len(df)
    dt = cfg["dt"]
    kpis = {"case": tag, "n_hours": T}

    # Voltage
    vmax = safe_col(df, "vmax")
    vmin = safe_col(df, "vmin")
    kpis["N_viol_over"]      = int(np.sum(vmax > cfg["V_max"]))
    kpis["N_viol_under"]     = int(np.sum(vmin < cfg["V_min"]))
    kpis["worst_Vmax_pu"]    = float(np.nanmax(vmax))
    kpis["worst_Vmin_pu"]    = float(np.nanmin(vmin))
    kpis["viol_severity_puh"]= float(
        np.sum(np.maximum(vmax - cfg["V_max"], 0))
        + np.sum(np.maximum(cfg["V_min"] - vmin, 0))
    )

    # Losses
    loss = safe_col(df, "losses_kw", fill=0.0)
    kpis["energy_loss_kwh"]  = float(np.nansum(loss) * dt)

    # PV
    avail = safe_col(df, "pv_actual_kw",    fill=0.0)
    deliv = safe_col(df, "pv_delivered_kw", fill=avail)
    kpis["pv_available_kwh"] = float(np.nansum(avail) * dt)
    kpis["pv_delivered_kwh"] = float(np.nansum(deliv) * dt)
    curt  = np.maximum(avail - deliv, 0.0)
    kpis["pv_curtailed_kwh"] = float(np.nansum(curt) * dt)
    denom = kpis["pv_available_kwh"]
    kpis["curtailment_pct"]  = (kpis["pv_curtailed_kwh"] / denom * 100.0
                                 if denom > 0 else 0.0)

    # BESS
    pbess = safe_col(df, "p_bess_kw", fill=0.0)
    kpis["bess_throughput_kwh"] = float(np.nansum(np.abs(pbess)) * dt / 2.0)
    soc   = safe_col(df, "soc", fill=cfg["soc_init"])
    kpis["soc_viol_hours"] = int(
        np.sum((soc < cfg["soc_min"] - 1e-4) | (soc > cfg["soc_max"] + 1e-4))
    )

    # Solver
    if "solve_ms" in df.columns:
        kpis["mean_solve_ms"] = float(df["solve_ms"].mean())
    if "mpc_status" in df.columns:
        kpis["infeasible_steps"] = int((df["mpc_status"] == "infeasible").sum())

    # Head energy (economics hook – tariff inputs plug in here later)
    head = safe_col(df, "head_p_kw", fill=0.0)
    kpis["feeder_import_kwh"]  = float(np.nansum(np.maximum(head,  0)) * dt)
    kpis["feeder_export_kwh"]  = float(np.nansum(np.maximum(-head, 0)) * dt)
    # ── ECONOMICS PLACEHOLDER ─────────────────────────────────────────────
    # To compute economic savings, insert tariff values here:
    #   import_tariff_per_kwh = ???    (e.g. 0.25 $/kWh)
    #   export_tariff_per_kwh = ???    (e.g. 0.08 $/kWh)
    #   bess_degradation_cost = ???    (e.g. 0.05 $/kWh throughput)
    #   cost_import = feeder_import_kwh * import_tariff_per_kwh
    #   revenue_export = feeder_export_kwh * export_tariff_per_kwh
    #   degradation_cost = bess_throughput_kwh * bess_degradation_cost
    kpis["economics_placeholder"] = "insert tariff data in next stage"
    # ──────────────────────────────────────────────────────────────────────

    return kpis

# ── Compute for each case ─────────────────────────────────────────────────
kpi_mpc  = compute_kpis(df_annual, "MPC Controlled",    CONFIG)
kpi_base = compute_kpis(df_base,   "DER Uncontrolled",  CONFIG)
kpi_list = [kpi_base, kpi_mpc]

if df_noder is not None:
    kpi_nd = compute_kpis(df_noder, "No DER Baseline", CONFIG)
    kpi_list = [kpi_nd] + kpi_list

# ── Print table ───────────────────────────────────────────────────────────
KPI_DISPLAY = [
    ("N_viol_over",         "Over-voltage hours"),
    ("N_viol_under",        "Under-voltage hours"),
    ("worst_Vmax_pu",       "Worst Vmax (pu)"),
    ("worst_Vmin_pu",       "Worst Vmin (pu)"),
    ("viol_severity_puh",   "Violation severity (pu·h)"),
    ("energy_loss_kwh",     "Annual losses (kWh)"),
    ("pv_available_kwh",    "PV available (kWh)"),
    ("pv_delivered_kwh",    "PV delivered (kWh)"),
    ("pv_curtailed_kwh",    "PV curtailed (kWh)"),
    ("curtailment_pct",     "Curtailment (%)"),
    ("bess_throughput_kwh", "BESS throughput (kWh)"),
    ("soc_viol_hours",      "SOC violation hours"),
    ("feeder_import_kwh",   "Feeder import (kWh)"),
    ("feeder_export_kwh",   "Feeder export (kWh)"),
]

col_names = [k["case"] for k in kpi_list]
header    = f"{'KPI':<32} " + "  ".join(f"{c:>22}" for c in col_names)
sep       = "─" * len(header)
print(f"\n{sep}\n{header}\n{sep}")
for key, label in KPI_DISPLAY:
    row = f"{label:<32} "
    for k in kpi_list:
        v = k.get(key, "N/A")
        row += f"  {str(v):>22}"
    print(row)
print(sep)

# ── Markdown table ────────────────────────────────────────────────────────
md_lines = ["| KPI | " + " | ".join(col_names) + " |",
            "|-----|" + "|".join(["---"] * len(col_names)) + "|"]
for key, label in KPI_DISPLAY:
    vals = [str(round(k.get(key, "N/A"), 4)) if isinstance(k.get(key), float)
            else str(k.get(key, "N/A"))
            for k in kpi_list]
    md_lines.append(f"| {label} | " + " | ".join(vals) + " |")

md_path = os.path.join(CONFIG["out_dir"], "mpc_summary_table.md")
with open(md_path, "w") as f:
    f.write("\n".join(md_lines))
print(f"\nMarkdown table saved: {md_path}")

# ── mpc_summary.json ─────────────────────────────────────────────────────
summary = {
    "meta": {
        "project": "Predictive Coordinated Volt-VAR and BESS Dispatch",
        "network": "IEEE LVTestCase",
        "solver":  "OSQP via CVXPY",
        "formulation": "QP with linear sensitivity prediction model",
        "horizon_H":   CONFIG["H"],
        "dt_h":        CONFIG["dt"],
        "T_hours":     CONFIG["T"],
    },
    "weights": {
        "w_V":  CONFIG["w_V"],
        "w_du": CONFIG["w_du"],
        "w_B":  CONFIG["w_B"],
        "w_C":  CONFIG["w_C"],
    },
    "constraints": {
        "V_min_pu":     CONFIG["V_min"],
        "V_max_pu":     CONFIG["V_max"],
        "S_inv_kva":    CONFIG["S_inv_kva"],
        "bess_kw_rated":CONFIG["bess_kw_rated"],
        "bess_kwh_rated":CONFIG["bess_kwh_rated"],
        "eta_ch":       CONFIG["eta_ch"],
        "eta_dis":      CONFIG["eta_dis"],
        "soc_min":      CONFIG["soc_min"],
        "soc_max":      CONFIG["soc_max"],
        "ramp_dP_max":  CONFIG["dP_bess_max"],
        "ramp_dQ_max":  CONFIG["dQ_pv_max"],
    },
    "kpis": {k["case"]: {kk: vv for kk, vv in k.items() if kk != "case"}
             for k in kpi_list},
    "runtime": {
        "mean_solve_ms":  kpi_mpc.get("mean_solve_ms", None),
        "infeasible_steps": kpi_mpc.get("infeasible_steps", None),
    },
}

with open(CONFIG["mpc_summary_json"], "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"JSON summary saved: {CONFIG['mpc_summary_json']}")
print("\n  ✓ All outputs complete.")